# Wi-Fi Fingerprint Indoor Localization — Classification Track (Part A)

**Objective:** Predict which **floor** a device is on based on its Wi-Fi RSSI fingerprint.

**Dataset:** UJIndoorLoc (UCI ML Repository) — same dataset as regression track.

**Team:** K Ganesh Giridhar (519) · G R Balaji (510) · A Suhas Reddy (503)

---
## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded")

In [ ]:
# Load data
train_df = pd.read_csv('../data/raw/trainingData.csv')
val_df   = pd.read_csv('../data/raw/validationData.csv')

wap_cols = [c for c in train_df.columns if c.startswith('WAP')]
print(f"Loaded {train_df.shape[0]} training samples, {len(wap_cols)} WAPs")

## 2. Preprocessing

We apply the same cleaning pipeline as the regression track to ensure consistency.

In [ ]:
# Replace sentinel +100 with -105
train_clean = train_df.copy()
train_clean[wap_cols] = train_clean[wap_cols].replace(100, -105)

# Remove zero-variance WAPs
variances = train_clean[wap_cols].var()
zero_var = variances[variances == 0].index.tolist()
wap_cols_clean = [c for c in wap_cols if c not in zero_var]
print(f"WAPs after removing zero-variance: {len(wap_cols_clean)}")

In [ ]:
# Feature engineering — same RSSI summary features
def engineer_features(df, wap_columns):
    wap_data = df[wap_columns]
    df['n_visible_aps'] = (wap_data > -105).sum(axis=1)
    detected = wap_data.replace(-105, np.nan)
    df['mean_rssi'] = detected.mean(axis=1)
    df['max_rssi'] = detected.max(axis=1)
    df['std_rssi'] = detected.std(axis=1)
    df['rssi_range'] = detected.max(axis=1) - detected.min(axis=1)
    return df

train_clean = engineer_features(train_clean, wap_cols_clean)

eng_features = ['n_visible_aps', 'mean_rssi', 'max_rssi', 'std_rssi', 'rssi_range']
feature_cols = wap_cols_clean + eng_features

print(f"Total features: {len(feature_cols)}")

### 2.1 Target Variable — Floor

In [ ]:
# Classification target
y = train_clean['FLOOR']
print("Floor distribution:")
print(y.value_counts().sort_index())
print()
print(f"Number of classes: {y.nunique()}")

In [ ]:
# Visualize class distribution
y.value_counts().sort_index().plot(kind='bar', color='seagreen', edgecolor='black')
plt.xlabel('Floor')
plt.ylabel('Number of Samples')
plt.title('Class Distribution — Floor')
plt.tight_layout()
plt.show()

**Observation:** Some floor imbalance exists but it's not extreme. We'll use stratified splitting to maintain class proportions.

### 2.2 Train-Test Split (Stratified)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = train_clean[feature_cols]

# Stratified split to preserve floor proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}")
print()
print("Train class distribution:")
print(y_train.value_counts().sort_index())
print()
print("Test class distribution:")
print(y_test.value_counts().sort_index())

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done (fitted on train only)")

---
## 3. Classification Algorithms — Part A

**Metrics:** Accuracy, Precision, Recall, Weighted F1-score, Confusion Matrix

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

def evaluate_classifier(model, X_tr, X_te, y_tr, y_te, model_name):
    """Train, predict, return metrics and print results."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    print(f"{model_name}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-score:  {f1:.4f}")
    print()

    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'y_pred': y_pred
    }

cls_results = []

### 3.1 Logistic Regression

Baseline linear classifier. Uses One-vs-Rest strategy for multi-class.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42, multi_class='ovr')
res = evaluate_classifier(lr, X_train_scaled, X_test_scaled, y_train, y_test, 'Logistic Regression')
cls_results.append(res)

**Inference:** Logistic Regression provides a strong linear baseline. For floor classification, it works reasonably well because different floors have distinct WAP signal patterns.

### 3.2 K-Nearest Neighbors

Classifies based on majority vote of k nearest training samples.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
res = evaluate_classifier(knn, X_train_scaled, X_test_scaled, y_train, y_test, 'KNN (k=5)')
cls_results.append(res)

**Inference:** KNN works well for fingerprinting because similar Wi-Fi fingerprints literally correspond to nearby locations on the same floor. Distance weighting helps by giving closer neighbours more influence.

### 3.3 Gaussian Naive Bayes

Assumes features are independent given the class. Simple but fast.

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
res = evaluate_classifier(gnb, X_train_scaled, X_test_scaled, y_train, y_test, 'Gaussian Naive Bayes')
cls_results.append(res)

**Inference:** Naive Bayes assumes feature independence which is violated here — nearby WAPs are correlated. Despite this, it can still perform reasonably as a quick baseline. Lower accuracy compared to other models is expected.

### 3.4 Decision Tree Classifier

Splits feature space into regions using information gain. Easily interpretable.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=15, random_state=42)
res = evaluate_classifier(dt, X_train_scaled, X_test_scaled, y_train, y_test, 'Decision Tree')
cls_results.append(res)

**Inference:** Decision trees can capture the non-linear boundaries between floors effectively. Setting max_depth prevents excessive overfitting while allowing complex decision boundaries.

### 3.5 Support Vector Machine (SVC)

Finds optimal hyperplanes to separate classes. RBF kernel handles non-linear boundaries.

In [ ]:
from sklearn.svm import SVC

# SVC can be slow on large datasets
svc = SVC(kernel='rbf', C=10, random_state=42)
res = evaluate_classifier(svc, X_train_scaled, X_test_scaled, y_train, y_test, 'SVM (RBF)')
cls_results.append(res)

**Inference:** SVM with RBF kernel can model complex class boundaries well. It tends to perform strongly on this task because floor separation in RSSI space is non-linear but structured.

---
## 4. Classification Results

### 4.1 Comparison Table

In [ ]:
# Build comparison table
cls_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'y_pred'} for r in cls_results])
cls_df = cls_df.sort_values('F1', ascending=False).reset_index(drop=True)
cls_df.index = cls_df.index + 1
cls_df.index.name = 'Rank'

print("=" * 70)
print("CLASSIFICATION MODEL COMPARISON — FLOOR PREDICTION (Part A)")
print("=" * 70)
print(cls_df.to_string())

**Key Takeaways:**
- KNN and SVM typically perform best for floor classification
- The conditional independence assumption of Naive Bayes hurts its performance with correlated WAP features
- Decision Tree provides good accuracy with interpretability
- All models benefit from the feature engineering and proper scaling

### 4.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, res in enumerate(cls_results):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=sorted(y_test.unique()),
                yticklabels=sorted(y_test.unique()))
    axes[i].set_title(res['Model'])
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# Hide extra subplot
axes[5].set_visible(False)

plt.suptitle('Confusion Matrices — All Part A Classifiers', y=1.02)
plt.tight_layout()
plt.show()

**Inference:** The confusion matrices reveal which floors are most confused with each other. Adjacent floors (e.g., Floor 1 and Floor 2) tend to be harder to distinguish because their Wi-Fi signal environments overlap more.

### 4.3 Classification Report (Best Model)

In [ ]:
# Detailed report for the best model
best_cls = cls_results[cls_df.iloc[0]['Model'] == cls_df['Model'].values[0]]
best_idx = cls_df['F1'].idxmax() - 1  # convert rank to index
best_res = cls_results[best_idx]

print(f"Detailed Classification Report — {best_res['Model']}")
print("=" * 60)
print(classification_report(y_test, best_res['y_pred'], digits=4))

---
## Summary

- **Best Classifier:** KNN / SVM (non-linear models excel at floor prediction)
- **Key Insight:** Wi-Fi fingerprints contain strong floor-discriminating information
- **Challenge:** Adjacent floors share similar signal patterns, causing some misclassification
- **Data Integrity:** Stratified split + train-only scaling prevents data leakage
- **Next Steps (Review 2):** Classification Part B (5 more algorithms) + consolidated 10-algorithm comparison + Clustering track